# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (sMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on PyMC3.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, StudentT

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.generics:GenericModel` has been moved to `pydantic.BaseModel`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.78068905  0.36552521  0.87686354  0.66933622 -0.58435374]
 [-0.42857405  0.94130083  0.22606379 -0.80299161 -0.71982247]
 [-0.98863965 -0.74037394 -0.22390461  0.85065     0.29363735]
 [ 0.28518366 -0.67014001 -0.82599151  0.41206398  0.21418783]
 [-0.89372586 -0.90426786 -0.66031751 -0.6194975  -0.22380608]
 [ 0.04298105 -0.04145425 -0.02789239  0.46834553  0.73136129]
 [-0.84627289 -0.07691493 -0.88140788  0.43073769  0.04886003]
 [ 0.93003108 -0.17940002 -0.74544174  0.22119454  0.438719  ]
 [ 0.81392962 -0.25748923  0.14460285  0.64185257 -0.02067909]
 [ 0.67976259 -0.11702144 -0.04751126 -0.12117445  0.53711729]]


In [4]:
# define action model
actions = {
    "a1": BayesianLogisticRegression(alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()]),
    "a2": BayesianLogisticRegression(alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()]),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a1', 'a1', 'a2', 'a2', 'a2', 'a2', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples)
# simulate context from environment
simulated_context = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1 0 1 1 0 1 0 1 1 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(X, actions=pred_actions, rewards=simulated_rewards)

ValidationError: 2 validation errors for BaseCmabBernoulli.update
actions
  Got multiple values for argument [type=multiple_argument_values, input_value=['a2', 'a2', 'a1', 'a1', ... 'a1', 'a1', 'a2', 'a2'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/multiple_argument_values
context
  Missing required argument [type=missing_argument, input_value=ArgsKwargs((CmabBernoulli... 0, 0, 1, 0, 1, 1, 0])}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.11/v/missing_argument